# SHWD Stage 2 Ablation Setup

This notebook initializes Stage 2 (Custom Module Ablation Study) for Safety Helmet Detection.
It automatically detects Kaggle input datasets & notebook outputs, extracts Top-2 weights (`yolo11s_best.pt`, `yolov8s_best.pt`), converts VOC2028 to YOLO format, and smoke-tests custom modules (`CoordConv`, `RepConv`, `BiFormer`, `Focal-EIoU`).

**Mounted Kaggle Inputs Handled:**
- Dataset: `VOC2028` (`/kaggle/input/datasets/hannhu4002/voc2028`)
- Dataset: `shwd-benchmark-code` (`/kaggle/input/datasets/hannhu4002/shwd-benchmark-code`)
- Notebook Output: `Structural Re-parameterized YOLO Architecture1`
- Notebook Output: `Structural Re-parameterized YOLO Architecture2`
- Notebook Output: `SHWD_Baseline_Consolidated_2` (contains `SHWD_Compact_Outputs.zip`)

In [1]:
# Step 1: Install Dependencies
from pathlib import Path
import sys
import subprocess

def find_file(name):
    candidates = [Path.cwd() / name, Path('/kaggle/working') / name]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(name))
    return next((p for p in candidates if p.exists()), None)

req_path = find_file('requirements_kaggle.txt')
if req_path:
    print('Installing from:', req_path)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', '-r', str(req_path)], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'onnx', 'onnxruntime-gpu', 'pandas', 'albumentations', 'opencv-python-headless'], check=True)

Installing from: /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/requirements_kaggle.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 31.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.

In [2]:
# Step 2: Automatic Flexible Path & Weight Resolution
from pathlib import Path
import os
import shutil
import zipfile
import subprocess
import sys

def find_dir_by_markers(dirname, required_children):
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob(dirname):
            if p.is_dir() and all((p / child).exists() for child in required_children):
                return p
    return None

def find_file_anywhere(name):
    roots = [Path.cwd(), Path('/kaggle/working'), Path('/kaggle/input')]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.exists():
            return direct
        matches = list(root.rglob(name))
        if matches:
            return matches[0]
    return None

# 1. Locate VOC2028 Dataset Root
DATASET_ROOT = find_dir_by_markers('VOC2028', ['Annotations', 'JPEGImages', 'ImageSets'])
print(' DATASET_ROOT =', DATASET_ROOT)

# 2. Resolve Compact Artifacts & Top-2 Model Weights
EXTRACT_DIR = Path('/kaggle/working/extracted_compact')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Look for SHWD_Compact_Outputs.zip inside SHWD_Baseline_Consolidated_2
compact_zip = find_file_anywhere('SHWD_Compact_Outputs.zip')
if compact_zip and compact_zip.exists():
    print(f' Found Compact ZIP at: {compact_zip}. Extracting...')
    with zipfile.ZipFile(compact_zip, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print(f' Extracted artifacts to: {EXTRACT_DIR}')

# Locate Compact Root (either extracted zip or directory)
COMPACT_ROOT = find_dir_by_markers('SHWD_Compact_Outputs', ['weights', 'master_benchmark_results.csv'])
if COMPACT_ROOT is None and (EXTRACT_DIR / 'master_benchmark_results.csv').exists():
    COMPACT_ROOT = EXTRACT_DIR
print(' COMPACT_ROOT =', COMPACT_ROOT)

# 3. Resolve Support Scripts
SCRIPT_PATH = find_file_anywhere('kaggle_shwd_baseline.py')
MODULE_PATH = find_file_anywhere('custom_ablation_modules.py')
AUG_PATH = find_file_anywhere('albumentations_hardcase_policy.py')

print(' SCRIPT_PATH =', SCRIPT_PATH)
print(' MODULE_PATH =', MODULE_PATH)
print(' AUG_PATH =', AUG_PATH)

# Verify code scripts & dataset exist
if not DATASET_ROOT or not SCRIPT_PATH or not MODULE_PATH:
    raise FileNotFoundError('Missing required VOC2028 dataset or python scripts! Check input attachments.')

# 4. Flexible Top-2 Model Weights Lookup (Zip Extraction or Raw Run Outputs)
TOP2_WEIGHTS = {}

# Try 1: Look in COMPACT_ROOT weights
if COMPACT_ROOT and (COMPACT_ROOT / 'weights').exists():
    w_dir = COMPACT_ROOT / 'weights'
    y11 = list(w_dir.glob('*yolo11s*best*.pt'))
    y8 = list(w_dir.glob('*yolov8s*best*.pt'))
    if y11:
        TOP2_WEIGHTS['yolo11s'] = y11[0]
    if y8:
        TOP2_WEIGHTS['yolov8s'] = y8[0]

# Try 2: Direct lookup in raw mounted notebook outputs if zip didn't have them
if 'yolo11s' not in TOP2_WEIGHTS or not TOP2_WEIGHTS['yolo11s'].exists():
    for p in Path('/kaggle/input').rglob('best.pt'):
        p_str = str(p).lower()
        if 'yolo11s' in p_str and ('architecture2' in p_str or 'architecture-2' in p_str):
            TOP2_WEIGHTS['yolo11s'] = p
            break

if 'yolov8s' not in TOP2_WEIGHTS or not TOP2_WEIGHTS['yolov8s'].exists():
    for p in Path('/kaggle/input').rglob('best.pt'):
        p_str = str(p).lower()
        if 'yolov8s' in p_str and ('architecture1' in p_str or 'architecture-1' in p_str):
            TOP2_WEIGHTS['yolov8s'] = p
            break

# Print weights status
print('\n--- Top-2 Champion Weights Found ---')
for name, path in TOP2_WEIGHTS.items():
    mb = round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else 0
    print(f'  - {name}: {path} (exists={path.exists()}, size={mb} MB)')
    if not path.exists():
        raise FileNotFoundError(f'Missing weights for {name}: {path}')

WORK_ROOT = Path('/kaggle/working/SHWD_STAGE2')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('\n✅ Stage 2 Setup Initialized Successfully at WORK_ROOT =', WORK_ROOT)

 DATASET_ROOT = /kaggle/input/datasets/hannhu4002/voc2028/VOC2028
 Found Compact ZIP at: /kaggle/input/notebooks/hannhu4002/shwd-baseline-consolidated-2/SHWD_Compact_Outputs.zip. Extracting...
 Extracted artifacts to: /kaggle/working/extracted_compact
 COMPACT_ROOT = /kaggle/working/extracted_compact
 SCRIPT_PATH = /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/kaggle_shwd_baseline.py
 MODULE_PATH = /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/custom_ablation_modules.py
 AUG_PATH = /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/albumentations_hardcase_policy.py

--- Top-2 Champion Weights Found ---
  - yolo11s: /kaggle/working/extracted_compact/weights/yolo11s_best.pt (exists=True, size=18.28 MB)
  - yolov8s: /kaggle/working/extracted_compact/weights/yolov8s_best.pt (exists=True, size=21.46 MB)

✅ Stage 2 Setup Initialized Successfully at WORK_ROOT = /kaggle/working/SHWD_STAGE2


In [3]:
# Step 3: Convert VOC2028 to YOLO format for Stage 2.
OUTPUT_DIR = Path('/kaggle/working/SHWD_YOLO_STAGE2')
cmd = [
    sys.executable, str(SCRIPT_PATH),
    '--mode', 'convert',
    '--dataset-root', str(DATASET_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--overwrite',
]
print('Executing command:', ' '.join(cmd))
subprocess.run(cmd, check=True)

Executing command: /usr/bin/python3 /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/kaggle_shwd_baseline.py --mode convert --dataset-root /kaggle/input/datasets/hannhu4002/voc2028/VOC2028 --output-dir /kaggle/working/SHWD_YOLO_STAGE2 --overwrite
{
  "dataset_root": "/kaggle/input/datasets/hannhu4002/voc2028/VOC2028",
  "output_dir": "/kaggle/working/SHWD_YOLO_STAGE2",
  "train_images": 6064,
  "test_images": 1517,
  "labels_written": 7581,
  "missing_images": 0,
  "missing_xml": 0,
  "invalid_boxes": 0,
  "ignored_objects": 3,
  "class_counts": {
    "hat": 9044,
    "person": 111514
  },
  "ignored_by_class": {
    "dog": 3
  },
  "dry_run": false
}


CompletedProcess(args=['/usr/bin/python3', '/kaggle/input/datasets/hannhu4002/shwd-benchmark-code/kaggle_shwd_baseline.py', '--mode', 'convert', '--dataset-root', '/kaggle/input/datasets/hannhu4002/voc2028/VOC2028', '--output-dir', '/kaggle/working/SHWD_YOLO_STAGE2', '--overwrite'], returncode=0)

In [4]:
# Step 4: Verify Conversion Report
import json

report_path = OUTPUT_DIR / 'conversion_report.json'
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2))
assert report['train_images'] == 6064, report['train_images']
assert report['test_images'] == 1517, report['test_images']
assert report['class_counts']['hat'] == 9044, report['class_counts']
assert report['class_counts']['person'] == 111514, report['class_counts']
assert report['ignored_by_class'].get('dog') == 3, report['ignored_by_class']
print('✅ Stage 2 dataset conversion verified successfully!')

{
  "dataset_root": "/kaggle/input/datasets/hannhu4002/voc2028/VOC2028",
  "output_dir": "/kaggle/working/SHWD_YOLO_STAGE2",
  "train_images": 6064,
  "test_images": 1517,
  "labels_written": 7581,
  "missing_images": 0,
  "missing_xml": 0,
  "invalid_boxes": 0,
  "ignored_objects": 3,
  "class_counts": {
    "hat": 9044,
    "person": 111514
  },
  "ignored_by_class": {
    "dog": 3
  },
  "dry_run": false
}
✅ Stage 2 dataset conversion verified successfully!


In [5]:
# Step 5: Copy Top-2 Weights to Working Directory
STAGE2_WEIGHTS = WORK_ROOT / 'weights'
STAGE2_WEIGHTS.mkdir(parents=True, exist_ok=True)
for name, src in TOP2_WEIGHTS.items():
    dst = STAGE2_WEIGHTS / src.name
    shutil.copy2(src, dst)
    print(f'Copied {name} -> {dst} ({round(dst.stat().st_size / (1024 * 1024), 2)} MB)')

Copied yolo11s -> /kaggle/working/SHWD_STAGE2/weights/yolo11s_best.pt (18.28 MB)
Copied yolov8s -> /kaggle/working/SHWD_STAGE2/weights/yolov8s_best.pt (21.46 MB)


In [6]:
# Step 6: Smoke-Test Custom Modules under Kaggle PyTorch
cmd = [sys.executable, str(MODULE_PATH)]
print('Running module test:', ' '.join(cmd))
subprocess.run(cmd, check=True)

Running module test: /usr/bin/python3 /kaggle/input/datasets/hannhu4002/shwd-benchmark-code/custom_ablation_modules.py
{'shape': (2, 24, 32, 32), 'repconv_fusion_max_diff': 2.384185791015625e-06}
{'focal_eiou': 0.20399829745292664}


CompletedProcess(args=['/usr/bin/python3', '/kaggle/input/datasets/hannhu4002/shwd-benchmark-code/custom_ablation_modules.py'], returncode=0)

In [7]:
# Step 7: Display Master Benchmark Leaderboard
import pandas as pd

csv_path = find_file_anywhere('master_benchmark_results.csv')
if csv_path and csv_path.exists():
    df = pd.read_csv(csv_path)
    cols = [c for c in ['model', 'map50', 'map50_95', 'ap50_hat', 'ap_hat', 'recall_hat', 'precision_hat', 'f1_hat', 'speed_inference_ms', 'onnx_latency_mean_ms', 'onnx_fps_mean', 'best_pt_mb'] if c in df.columns]
    print(' Master Stage 1 Baseline Matrix:')
    display(df[cols].sort_values(['map50_95', 'map50'], ascending=False))
else:
    print(' Master CSV not found, setup ready for Stage 2 ablation training.')

 Master Stage 1 Baseline Matrix:


,model,map50,map50_95,ap50_hat,ap_hat,recall_hat,precision_hat,f1_hat,speed_inference_ms,onnx_latency_mean_ms,onnx_fps_mean,best_pt_mb
0,yolo11s.pt,0.947379,0.625415,0.940598,0.742646,0.903456,0.911165,0.907294,6.524133,144.5346,6.92,18.278
1,yolov8s.pt,0.948927,0.622085,0.942811,0.737160,0.906199,0.917859,0.911991,6.105510,187.6546,5.33,21.465
2,yolov10s.pt,0.943913,0.621872,0.933565,0.734777,0.891388,0.918822,0.904897,6.227032,147.7368,6.77,15.749
3,yolov10n.pt,0.932999,0.603535,0.929662,0.718302,0.871340,0.915824,0.893028,2.663596,90.6514,11.03,5.475
4,yolov8n.pt,0.932052,0.602583,0.924177,0.718098,0.871640,0.904821,0.887921,2.855711,63.4755,15.75,5.949
5,yolo11n.pt,0.931852,0.602088,0.923260,0.715203,0.865058,0.930009,0.896358,3.182678,68.2158,14.66,5.207


## Next Execution Step: Stage 2 Custom Ablation Experiments

After this setup notebook passes, begin custom ablation training using **`yolo11s_best.pt`** and **`yolov8s_best.pt`** as controls.

**Ablation Progression:**
1. **A0**: Re-evaluate `yolo11s_best.pt` & `yolov8s_best.pt` as control baselines.
2. **A1**: Apply hard-case online augmentation (Albumentations: RandomShadow, HSV shift, Cutout).
3. **A2**: Inject **CoordConv** in stem/neck.
4. **A3**: Inject **RepConv / RepC3** and verify `switch_to_deploy()` fusion.
5. **A4**: Apply **Focal-EIoU Loss** for bounding box regression + alpha-focal classification for `hat`.
6. **A5**: Inject **BiFormer Attention** if small/occluded helmets require fine-grained routing.
7. **A6**: Train final combined candidate.